In [ ]:
# Uncomment if running in a fresh Colab runtime.
# %pip install -q pandas numpy scikit-learn scipy requests joblib pyarrow pybaseball xgboost lightgbm google-cloud-storage matplotlib

from __future__ import annotations

import json
import math
import os
import time
import warnings
from dataclasses import dataclass
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import requests

from scipy.stats import norm
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge, PoissonRegressor
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    mean_poisson_deviance,
    r2_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBClassifier, XGBRegressor
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

try:
    from lightgbm import LGBMClassifier, LGBMRegressor
    HAS_LIGHTGBM = True
except Exception:
    HAS_LIGHTGBM = False

try:
    import joblib
    HAS_JOBLIB = True
except Exception:
    HAS_JOBLIB = False

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 200)
print("Setup complete")
print("XGBoost:", HAS_XGBOOST, "LightGBM:", HAS_LIGHTGBM, "joblib:", HAS_JOBLIB)


Setup complete
XGBoost: True LightGBM: True joblib: True


In [ ]:
# -----------------------
# Core config
# -----------------------
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
PREDICTIONS_DIR = DATA_DIR / "predictions"
for p in [DATA_DIR, MODELS_DIR, PREDICTIONS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TODAY = pd.Timestamp.utcnow().date()
START_DATE = os.getenv("START_DATE", "2023-01-01")
END_DATE = os.getenv("END_DATE", (TODAY + timedelta(days=2)).isoformat())
DAYS_FORWARD_FOR_SCORING = int(os.getenv("DAYS_FORWARD", "2"))

SPORT_KEY = os.getenv("ODDS_SPORT_KEY", "baseball_mlb")
ODDS_REGIONS = os.getenv("ODDS_REGIONS", "us")
ODDS_MARKETS = os.getenv("ODDS_MARKETS", "h2h,spreads,totals")
ODDS_FORMAT = os.getenv("ODDS_FORMAT", "american")
ODDS_API_KEY = os.getenv("ODDS_API_KEY", "")  # Use getpass below if blank.

FETCH_SCHEDULE = True
FETCH_BOXSCORES = True
FETCH_ODDS = True
FETCH_STATCAST = True  # Set True when ready. Statcast pulls can be slow.

STATCAST_CHUNK_DAYS = 7
API_SLEEP_SECONDS = 0.15

MIN_TRAIN_DATE = "2024-01-01"
#MIN_TRAIN_DATE = "2023-01-01"
TEST_FRAC = 0.20
RANDOM_STATE = 42

# Betting thresholds. Tune conservatively at first.
MIN_EDGE_MONEYLINE = 0.02
MIN_EDGE_TOTALS = 0.04
MIN_EDGE_RUNLINE = 0.04
MIN_EV = 0.00

# Export flags default off.
APPROVE_MONEYLINE_EXPORT = False
APPROVE_TOTALS_EXPORT = False
APPROVE_MARGIN_EXPORT = False

print({
    "START_DATE": START_DATE,
    "END_DATE": END_DATE,
    "ODDS_MARKETS": ODDS_MARKETS,
    "FETCH_STATCAST": FETCH_STATCAST,
})


{'START_DATE': '2023-01-01', 'END_DATE': '2026-06-14', 'ODDS_MARKETS': 'h2h,spreads,totals', 'FETCH_STATCAST': True}


In [ ]:
def normalize_team_name(x: Any) -> str:
    if x is None or pd.isna(x):
        return ""
    s = str(x).strip().lower()
    repl = {
        ".": "", "'": "", "&": "and",
        "  ": " ",
    }
    for a, b in repl.items():
        s = s.replace(a, b)
    s = " ".join(s.split())
    aliases = {
        "oakland athletics": "athletics",
        "the athletics": "athletics",
        "la dodgers": "los angeles dodgers",
        "la angels": "los angeles angels",
        "d-backs": "arizona diamondbacks",
        "diamondbacks": "arizona diamondbacks",
        "white sox": "chicago white sox",
        "red sox": "boston red sox",
    }
    return aliases.get(s, s)


def american_to_implied_prob(price: Any) -> float:
    if price is None or pd.isna(price):
        return np.nan
    p = float(price)
    if p > 0:
        return 100.0 / (p + 100.0)
    return abs(p) / (abs(p) + 100.0)


def american_profit_per_unit(price: Any) -> float:
    if price is None or pd.isna(price):
        return np.nan
    p = float(price)
    if p > 0:
        return p / 100.0
    return 100.0 / abs(p)


def expected_value_per_unit(model_prob: float, american_price: float) -> float:
    if pd.isna(model_prob) or pd.isna(american_price):
        return np.nan
    profit = american_profit_per_unit(american_price)
    return model_prob * profit - (1.0 - model_prob)


def no_vig_two_way_prob(price_a: Any, price_b: Any) -> tuple[float, float]:
    pa = american_to_implied_prob(price_a)
    pb = american_to_implied_prob(price_b)
    if pd.isna(pa) or pd.isna(pb) or (pa + pb) <= 0:
        return np.nan, np.nan
    return pa / (pa + pb), pb / (pa + pb)


def daterange_chunks(start_date: str | date, end_date: str | date, chunk_days: int):
    start = pd.to_datetime(start_date).date()
    end = pd.to_datetime(end_date).date()
    cur = start
    while cur <= end:
        chunk_end = min(cur + timedelta(days=chunk_days - 1), end)
        yield cur, chunk_end
        cur = chunk_end + timedelta(days=1)


def rmse(y_true, pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, pred)))


def evaluate_regression(y_true, pred, allow_poisson: bool = False) -> dict[str, float]:
    yt = np.asarray(y_true, dtype=float)
    pr = np.asarray(pred, dtype=float)
    mask = np.isfinite(yt) & np.isfinite(pr)
    yt = yt[mask]
    pr = pr[mask]
    out = {
        "mae": float(mean_absolute_error(yt, pr)),
        "rmse": float(np.sqrt(mean_squared_error(yt, pr))),
        "r2": float(r2_score(yt, pr)),
        "avg_pred": float(np.mean(pr)),
        "actual_mean": float(np.mean(yt)),
    }
    if allow_poisson:
        pr_pos = np.clip(pr, 1e-6, None)
        yt_pos = np.clip(yt, 0, None)
        try:
            out["poisson_deviance"] = float(mean_poisson_deviance(yt_pos, pr_pos))
        except Exception:
            out["poisson_deviance"] = np.nan
    return out


def evaluate_binary(y_true, prob) -> dict[str, float]:
    yt = np.asarray(y_true, dtype=int)
    pr = np.asarray(prob, dtype=float)
    pr = np.clip(pr, 1e-6, 1 - 1e-6)
    return {
        "n_test": int(len(yt)),
        "avg_pred": float(np.mean(pr)),
        "actual_rate": float(np.mean(yt)),
        "log_loss": float(log_loss(yt, pr)),
        "brier": float(brier_score_loss(yt, pr)),
        "roc_auc": float(roc_auc_score(yt, pr)) if len(np.unique(yt)) == 2 else np.nan,
        "accuracy_50pct": float(accuracy_score(yt, pr >= 0.5)),
    }


def chronological_split(df: pd.DataFrame, test_frac: float = 0.20) -> tuple[pd.DataFrame, pd.DataFrame]:
    sort_cols = [c for c in ["official_date", "game_datetime_utc", "game_pk"] if c in df.columns]
    d = df.sort_values(sort_cols).reset_index(drop=True)
    split_idx = int(len(d) * (1 - test_frac))
    return d.iloc[:split_idx].copy(), d.iloc[split_idx:].copy()

print("Utility functions ready")


Utility functions ready


In [ ]:
def fetch_mlb_schedule(start_date: str, end_date: str, game_type: str = "R", chunk_days: int = 30) -> pd.DataFrame:
    """Fetch MLB games from statsapi.mlb.com schedule endpoint."""
    rows = []
    for s, e in daterange_chunks(start_date, end_date, chunk_days):
        url = "https://statsapi.mlb.com/api/v1/schedule"
        params = {
            "sportId": 1,
            "startDate": s.isoformat(),
            "endDate": e.isoformat(),
            "gameTypes": game_type,
            "hydrate": "probablePitcher,linescore",
        }
        r = requests.get(url, params=params, timeout=45)
        r.raise_for_status()
        data = r.json()
        for dblock in data.get("dates", []) or []:
            for game in dblock.get("games", []) or []:
                teams = game.get("teams", {}) or {}
                home = teams.get("home", {}) or {}
                away = teams.get("away", {}) or {}
                home_team = (home.get("team", {}) or {})
                away_team = (away.get("team", {}) or {})
                status = game.get("status", {}) or {}
                hp = home.get("probablePitcher", {}) or {}
                ap = away.get("probablePitcher", {}) or {}
                rows.append({
                    "game_pk": game.get("gamePk"),
                    "official_date": game.get("officialDate") or dblock.get("date"),
                    "game_datetime_utc": game.get("gameDate"),
                    "game_type": game.get("gameType"),
                    "detailed_state": status.get("detailedState"),
                    "abstract_state": status.get("abstractGameState"),
                    "home_team_id": home_team.get("id"),
                    "home_team_name": home_team.get("name"),
                    "home_team_norm": normalize_team_name(home_team.get("name")),
                    "away_team_id": away_team.get("id"),
                    "away_team_name": away_team.get("name"),
                    "away_team_norm": normalize_team_name(away_team.get("name")),
                    "home_score": home.get("score"),
                    "away_score": away.get("score"),
                    "home_probable_pitcher_id": hp.get("id"),
                    "home_probable_pitcher_name": hp.get("fullName"),
                    "away_probable_pitcher_id": ap.get("id"),
                    "away_probable_pitcher_name": ap.get("fullName"),
                })
        print(f"schedule {s} to {e}: cumulative rows={len(rows)}")
        time.sleep(API_SLEEP_SECONDS)
    df = pd.DataFrame(rows).drop_duplicates("game_pk", keep="last")
    if not df.empty:
        df["official_date"] = pd.to_datetime(df["official_date"], errors="coerce")
        df["game_datetime_utc"] = pd.to_datetime(df["game_datetime_utc"], errors="coerce", utc=True)
        for c in ["home_score", "away_score", "home_team_id", "away_team_id", "home_probable_pitcher_id", "away_probable_pitcher_id"]:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")
        df["is_final"] = df["abstract_state"].eq("Final") | df["detailed_state"].astype(str).str.lower().isin(["final", "completed early"])
        df["target_home_win"] = np.where(df["is_final"] & df["home_score"].notna() & df["away_score"].notna(), (df["home_score"] > df["away_score"]).astype(float), np.nan)
        df["target_total_runs"] = np.where(df["is_final"], df["home_score"] + df["away_score"], np.nan)
        df["target_home_margin"] = np.where(df["is_final"], df["home_score"] - df["away_score"], np.nan)
    return df


def fetch_mlb_boxscore(game_pk: int) -> dict:
    url = f"https://statsapi.mlb.com/api/v1/game/{int(game_pk)}/boxscore"
    r = requests.get(url, timeout=45)
    r.raise_for_status()
    return r.json()


def parse_boxscore_team_rows(game_row: pd.Series, box: dict) -> list[dict]:
    """Parse team-level batting/pitching totals from MLB boxscore JSON."""
    out = []
    teams = box.get("teams", {}) or {}
    for side in ["home", "away"]:
        t = teams.get(side, {}) or {}
        team_meta = t.get("team", {}) or {}
        batting = ((t.get("teamStats", {}) or {}).get("batting", {}) or {})
        pitching = ((t.get("teamStats", {}) or {}).get("pitching", {}) or {})
        row = {
            "game_pk": game_row.get("game_pk"),
            "official_date": game_row.get("official_date"),
            "game_datetime_utc": game_row.get("game_datetime_utc"),
            "team_side": side,
            "team_id": team_meta.get("id") or game_row.get(f"{side}_team_id"),
            "team_name": team_meta.get("name") or game_row.get(f"{side}_team_name"),
            "team_norm": normalize_team_name(team_meta.get("name") or game_row.get(f"{side}_team_name")),
            "opponent_team_id": game_row.get("away_team_id" if side == "home" else "home_team_id"),
            "opponent_team_name": game_row.get("away_team_name" if side == "home" else "home_team_name"),
            "runs_for": game_row.get("home_score" if side == "home" else "away_score"),
            "runs_against": game_row.get("away_score" if side == "home" else "home_score"),
        }
        for k, v in batting.items():
            row[f"box_bat_{k}"] = v
        for k, v in pitching.items():
            row[f"box_pitch_{k}"] = v
        out.append(row)
    return out


def fetch_boxscores_for_games(games_df: pd.DataFrame, only_final: bool = True, limit: int | None = None) -> pd.DataFrame:
    candidates = games_df.copy()
    if only_final and "is_final" in candidates.columns:
        candidates = candidates[candidates["is_final"].eq(True)]
    if limit:
        candidates = candidates.head(limit)
    rows = []
    for i, (_, g) in enumerate(candidates.iterrows(), start=1):
        try:
            box = fetch_mlb_boxscore(int(g["game_pk"]))
            rows.extend(parse_boxscore_team_rows(g, box))
        except Exception as exc:
            print(f"boxscore failed game_pk={g.get('game_pk')}: {exc}")
        if i % 100 == 0:
            print(f"boxscores processed {i}/{len(candidates)}")
        time.sleep(API_SLEEP_SECONDS)
    return pd.DataFrame(rows)

print("MLB schedule/boxscore clients ready")


MLB schedule/boxscore clients ready


In [ ]:
def fetch_odds_events(api_key: str, sport_key: str = SPORT_KEY, regions: str = ODDS_REGIONS,
                      markets: str = ODDS_MARKETS, odds_format: str = ODDS_FORMAT) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Fetch odds from The Odds API and return events + long outcome snapshots."""
    if not api_key:
        raise ValueError("ODDS_API_KEY is blank. Set env var or enter key before running.")
    url = f"https://api.the-odds-api.com/v4/sports/{sport_key}/odds"
    params = {
        "apiKey": api_key,
        "regions": regions,
        "markets": markets,
        "oddsFormat": odds_format,
    }
    r = requests.get(url, params=params, timeout=60)
    print("Odds API status:", r.status_code, "remaining:", r.headers.get("x-requests-remaining"), "used:", r.headers.get("x-requests-used"))
    r.raise_for_status()
    data = r.json()
    fetched_at = pd.Timestamp.utcnow().isoformat()
    event_rows = []
    snap_rows = []
    for ev in data:
        event_id = ev.get("id")
        event_rows.append({
            "event_id": event_id,
            "sport_key": ev.get("sport_key"),
            "sport_title": ev.get("sport_title"),
            "commence_time_utc": ev.get("commence_time"),
            "home_team": ev.get("home_team"),
            "away_team": ev.get("away_team"),
            "home_team_norm": normalize_team_name(ev.get("home_team")),
            "away_team_norm": normalize_team_name(ev.get("away_team")),
            "fetched_at_utc": fetched_at,
        })
        for book in ev.get("bookmakers", []) or []:
            for market in book.get("markets", []) or []:
                mkey = market.get("key")
                for outcome in market.get("outcomes", []) or []:
                    snap_rows.append({
                        "fetched_at_utc": fetched_at,
                        "event_id": event_id,
                        "sport_key": ev.get("sport_key"),
                        "commence_time_utc": ev.get("commence_time"),
                        "home_team": ev.get("home_team"),
                        "away_team": ev.get("away_team"),
                        "home_team_norm": normalize_team_name(ev.get("home_team")),
                        "away_team_norm": normalize_team_name(ev.get("away_team")),
                        "bookmaker_key": book.get("key"),
                        "bookmaker_title": book.get("title"),
                        "bookmaker_last_update_utc": book.get("last_update"),
                        "market_key": mkey,
                        "outcome_name": outcome.get("name"),
                        "outcome_name_norm": normalize_team_name(outcome.get("name")),
                        "outcome_price": outcome.get("price"),
                        "outcome_point": outcome.get("point"),
                        "outcome_description": outcome.get("description"),
                    })
    events = pd.DataFrame(event_rows)
    odds = pd.DataFrame(snap_rows)
    for df in [events, odds]:
        if not df.empty and "commence_time_utc" in df.columns:
            df["commence_time_utc"] = pd.to_datetime(df["commence_time_utc"], utc=True, errors="coerce")
    return events, odds

print("Odds API client ready")


Odds API client ready


In [ ]:
def fetch_statcast_pybaseball(start_date: str, end_date: str, chunk_days: int = 7) -> pd.DataFrame:
    """Fetch pitch-level Statcast using pybaseball.statcast. This is inline and does not use repo scripts."""
    try:
        from pybaseball import statcast
    except Exception as exc:
        raise ImportError("pybaseball is required for Statcast pulls. Run `%pip install pybaseball`.") from exc

    frames = []
    for s, e in daterange_chunks(start_date, end_date, chunk_days):
        print(f"Fetching Statcast {s} to {e}")
        try:
            chunk = statcast(start_dt=s.isoformat(), end_dt=e.isoformat())
            if chunk is not None and len(chunk):
                frames.append(chunk)
                print("  rows", len(chunk))
        except Exception as exc:
            print(f"  Statcast failed {s} to {e}: {exc}")
        time.sleep(API_SLEEP_SECONDS)
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    df = df.drop_duplicates()
    return df

print("Statcast client ready")


Statcast client ready


In [ ]:
if FETCH_SCHEDULE:
    games = fetch_mlb_schedule(START_DATE, END_DATE, game_type="R", chunk_days=30)
else:
    games = pd.DataFrame()

print("games", games.shape)
if not games.empty:
    print(games[["official_date", "game_datetime_utc", "away_team_name", "home_team_name", "detailed_state", "abstract_state", "home_score", "away_score"]].head())
    print("date range", games["official_date"].min(), games["official_date"].max())
    print(games["abstract_state"].value_counts(dropna=False))


schedule 2023-01-01 to 2023-01-30: cumulative rows=0
schedule 2023-01-31 to 2023-03-01: cumulative rows=0
schedule 2023-03-02 to 2023-03-31: cumulative rows=20
schedule 2023-04-01 to 2023-04-30: cumulative rows=438
schedule 2023-05-01 to 2023-05-30: cumulative rows=842
schedule 2023-05-31 to 2023-06-29: cumulative rows=1239
schedule 2023-06-30 to 2023-07-29: cumulative rows=1602
schedule 2023-07-30 to 2023-08-28: cumulative rows=2009
schedule 2023-08-29 to 2023-09-27: cumulative rows=2417
schedule 2023-09-28 to 2023-10-27: cumulative rows=2476
schedule 2023-10-28 to 2023-11-26: cumulative rows=2476
schedule 2023-11-27 to 2023-12-26: cumulative rows=2476
schedule 2023-12-27 to 2024-01-25: cumulative rows=2476
schedule 2024-01-26 to 2024-02-24: cumulative rows=2476
schedule 2024-02-25 to 2024-03-25: cumulative rows=2478
schedule 2024-03-26 to 2024-04-24: cumulative rows=2860
schedule 2024-04-25 to 2024-05-24: cumulative rows=3263
schedule 2024-05-25 to 2024-06-23: cumulative rows=3666
sc

In [ ]:
if FETCH_BOXSCORES and not games.empty:
    box_team_game = fetch_boxscores_for_games(games, only_final=True, limit=None)
else:
    box_team_game = pd.DataFrame()

print("box_team_game", box_team_game.shape)
display(box_team_game.head())


boxscores processed 100/8320
boxscores processed 200/8320
boxscores processed 300/8320
boxscores processed 400/8320
boxscores processed 500/8320
boxscores processed 600/8320
boxscores processed 700/8320
boxscores processed 800/8320
boxscores processed 900/8320
boxscores processed 1000/8320
boxscores processed 1100/8320
boxscores processed 1200/8320
boxscores processed 1300/8320
boxscores processed 1400/8320
boxscores processed 1500/8320
boxscores processed 1600/8320
boxscores processed 1700/8320
boxscores processed 1800/8320
boxscores processed 1900/8320
boxscores processed 2000/8320
boxscores processed 2100/8320
boxscores processed 2200/8320
boxscores processed 2300/8320
boxscores processed 2400/8320
boxscores processed 2500/8320
boxscores processed 2600/8320
boxscores processed 2700/8320
boxscores processed 2800/8320
boxscores processed 2900/8320
boxscores processed 3000/8320
boxscores processed 3100/8320
boxscores processed 3200/8320
boxscores processed 3300/8320
boxscores processed

,game_pk,official_date,game_datetime_utc,team_side,team_id,team_name,team_norm,opponent_team_id,opponent_team_name,runs_for,runs_against,box_bat_flyOuts,box_bat_groundOuts,box_bat_airOuts,box_bat_runs,box_bat_doubles,box_bat_triples,box_bat_homeRuns,box_bat_strikeOuts,box_bat_baseOnBalls,box_bat_intentionalWalks,box_bat_hits,box_bat_hitByPitch,box_bat_avg,box_bat_atBats,box_bat_obp,box_bat_slg,box_bat_ops,box_bat_caughtStealing,box_bat_stolenBases,box_bat_stolenBasePercentage,box_bat_groundIntoDoublePlay,box_bat_groundIntoTriplePlay,box_bat_plateAppearances,box_bat_totalBases,box_bat_rbi,box_bat_leftOnBase,box_bat_sacBunts,box_bat_sacFlies,box_bat_catchersInterference,box_bat_pickoffs,box_bat_atBatsPerHomeRun,box_bat_popOuts,box_bat_lineOuts,box_pitch_flyOuts,box_pitch_groundOuts,box_pitch_airOuts,box_pitch_runs,box_pitch_doubles,box_pitch_triples,box_pitch_homeRuns,box_pitch_strikeOuts,box_pitch_baseOnBalls,box_pitch_intentionalWalks,box_pitch_hits,box_pitch_hitByPitch,box_pitch_atBats,box_pitch_obp,box_pitch_caughtStealing,box_pitch_stolenBases,box_pitch_stolenBasePercentage,box_pitch_caughtStealingPercentage,box_pitch_numberOfPitches,box_pitch_era,box_pitch_inningsPitched,box_pitch_saveOpportunities,box_pitch_earnedRuns,box_pitch_whip,box_pitch_battersFaced,box_pitch_outs,box_pitch_completeGames,box_pitch_shutouts,box_pitch_pitchesThrown,box_pitch_balls,box_pitch_strikes,box_pitch_strikePercentage,box_pitch_hitBatsmen,box_pitch_balks,box_pitch_wildPitches,box_pitch_pickoffs,box_pitch_groundOutsToAirouts,box_pitch_rbi,box_pitch_pitchesPerInning,box_pitch_runsScoredPer9,box_pitch_homeRunsPer9,box_pitch_inheritedRunners,box_pitch_inheritedRunnersScored,box_pitch_catchersInterference,box_pitch_sacBunts,box_pitch_sacFlies,box_pitch_passedBall,box_pitch_popOuts,box_pitch_lineOuts
0,718780,2023-03-30,2023-03-30 17:05:00+00:00,home,120,Washington Nationals,washington nationals,144,Atlanta Braves,2.0,7.0,4,13,8,2,1,0,0,5,4,0,8,0,.242,33,.316,.273,.589,0,0,.---,1,0,38,9,2,24,0,1,0,0,-.--,2,2,5,12,9,7,2,0,0,7,6,0,12,0,40,.391,0,2,1.000,.000,177,4.00,9.0,0,4,2.00,46,27,0,0,177.0,69,108,.610,0,0,1,0,1.33,5,19.67,7.00,0.00,0,0,0,0,0,0,3,1
1,718780,2023-03-30,2023-03-30 17:05:00+00:00,away,144,Atlanta Braves,atlanta braves,120,Washington Nationals,7.0,2.0,5,12,9,7,2,0,0,7,6,0,12,0,.300,40,.391,.350,.741,0,2,1.000,1,0,46,14,5,23,0,0,0,0,-.--,3,1,4,13,8,2,1,0,0,5,4,0,8,0,33,.316,0,0,.---,.---,143,2.00,9.0,0,2,1.33,38,27,0,0,143.0,52,91,.640,0,0,0,0,1.63,2,15.89,2.00,0.00,0,0,0,0,1,0,2,2
2,718781,2023-03-30,2023-03-30 17:05:00+00:00,home,147,New York Yankees,new york yankees,137,San Francisco Giants,5.0,0.0,1,5,3,5,0,0,2,16,2,0,8,0,.250,32,.294,.438,.732,0,2,1.000,0,0,34,14,5,13,0,0,0,0,16.00,2,0,2,7,3,0,0,0,0,16,3,0,4,0,30,.212,0,1,1.000,.000,150,0.00,9.0,0,0,0.78,33,27,0,1,150.0,57,93,.620,0,1,0,0,2.33,0,16.67,0.00,0.00,0,0,0,0,0,0,1,0
3,718781,2023-03-30,2023-03-30 17:05:00+00:00,away,137,San Francisco Giants,san francisco giants,147,New York Yankees,0.0,5.0,2,7,3,0,0,0,0,16,3,0,4,0,.133,30,.212,.133,.345,0,1,1.000,1,0,33,4,0,14,0,0,0,0,-.--,1,0,1,5,3,5,0,0,2,16,2,0,8,0,32,.294,0,2,1.000,.000,126,5.63,8.0,0,5,1.25,34,24,0,0,126.0,37,89,.710,0,0,0,0,1.67,5,15.75,5.63,2.25,0,0,0,0,0,0,2,0
4,718782,2023-03-30,2023-03-30 18:10:00+00:00,home,111,Boston Red Sox,boston red sox,110,Baltimore Orioles,9.0,10.0,2,12,4,9,2,1,0,9,3,0,11,2,.314,35,.390,.429,.819,0,0,.---,2,0,41,15,8,15,0,1,0,0,-.--,2,0,2,7,9,10,4,0,2,8,9,0,15,1,38,.521,0,5,1.000,.000,191,10.00,9.0,0,10,2.67,49,27,0,0,191.0,74,117,.610,1,0,2,0,0.78,9,21.22,10.00,2.00,0,0,0,1,0,0,2,5


In [ ]:
# If ODDS_API_KEY is blank and you are in a notebook, uncomment this:
# import getpass
# ODDS_API_KEY = getpass.getpass("The Odds API key: ")

if FETCH_ODDS and ODDS_API_KEY:
    odds_events, odds_snapshots = fetch_odds_events(ODDS_API_KEY, markets=ODDS_MARKETS)
else:
    print("Skipping odds fetch because FETCH_ODDS is False or ODDS_API_KEY is blank.")
    odds_events, odds_snapshots = pd.DataFrame(), pd.DataFrame()

print("odds_events", odds_events.shape, "odds_snapshots", odds_snapshots.shape)
if not odds_snapshots.empty:
    display(odds_snapshots.groupby("market_key").agg(rows=("event_id", "size"), events=("event_id", "nunique")).reset_index())


Skipping odds fetch because FETCH_ODDS is False or ODDS_API_KEY is blank.
odds_events (0, 0) odds_snapshots (0, 0)


In [ ]:
%pip install pybaseball

In [ ]:
if FETCH_STATCAST:
    statcast_raw = fetch_statcast_pybaseball(START_DATE, END_DATE, chunk_days=STATCAST_CHUNK_DAYS)
else:
    statcast_raw = pd.DataFrame()

print("statcast_raw", statcast_raw.shape)
display(statcast_raw.head())


Fetching Statcast 2023-01-01 to 2023-01-07
This is a large query, it may take a moment to complete
Skipping offseason dates


0it [00:00, ?it/s]

Fetching Statcast 2023-01-08 to 2023-01-14
This is a large query, it may take a moment to complete


Skipping offseason dates


0it [00:00, ?it/s]


...

100%|██████████| 7/7 [00:06<00:00,  1.12it/s]


  rows 18340
Fetching Statcast 2026-06-14 to 2026-06-14
This is a large query, it may take a moment to complete


100%|██████████| 1/1 [00:00<00:00,  8.89it/s]


statcast_raw (2645941, 119)


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,tfs_deprecated,tfs_zulu_deprecated,umpire,sv_id,vx0,vy0,vz0,ax,ay,az,sz_top,sz_bot,hit_distance_sc,launch_speed,launch_angle,effective_speed,release_spin_rate,release_extension,game_pk,fielder_2,fielder_3,fielder_4,fielder_5,fielder_6,fielder_7,fielder_8,fielder_9,release_pos_y,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,pitch_name,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,miss_distance,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,SL,2023-03-18,84.7,-2.16,5.5,"Wallace, Jacob",686668,686608,force_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,9,"Brenton Doyle grounds into a force out, third ...",S,R,R,COL,KC,X,5,ground_ball,1,1,2023,0.78,-0.18,0.300694,1.855843,669911,<NA>,666134,2,9,Bot,100.33,162.86,<NA>,<NA>,<NA>,<NA>,4.185992,-123.367264,-1.825071,7.102894,24.276703,-33.89723,3.49,1.601,80,73.8,8,84.7,2999,6.2,733583,682515,680769,665834,689374,686475,674646,683031,687614,54.25,<NA>,<NA>,0.0,<NA>,0,0,2,82,3,Slider,5,8,5,8,8,5,5,8,Standard,Standard,51,-0.046,-0.301,<NA>,<NA>,<NA>,<NA>,0.301,88.0,-3,-3,0.046,0.046,24,25,25,25,1,4,<NA>,<NA>,<NA>,<NA>,3.35,-0.78,-0.78,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,FC,2023-03-18,91.4,-2.03,5.62,"Wallace, Jacob",686668,686608,NaN,ball,<NA>,<NA>,<NA>,<NA>,14,NaN,S,R,R,COL,KC,B,<NA>,NaN,0,1,2023,0.46,0.5,2.466818,1.229679,669911,<NA>,666134,2,9,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,10.359331,-132.664357,-6.290476,3.355396,27.370295,-25.062908,3.49,1.601,<NA>,<NA>,<NA>,91.7,2797,6.5,733583,682515,680769,665834,689374,686475,674646,683031,687614,53.96,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,82,2,Cutter,5,8,5,8,8,5,5,8,Standard,Standard,146,0.001,0.039,<NA>,<NA>,<NA>,<NA>,-0.039,<NA>,-3,-3,0.045,0.045,24,25,25,25,1,4,<NA>,<NA>,<NA>,<NA>,2.23,-0.46,-0.46,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,FF,2023-03-18,98.7,-1.75,5.86,"Wallace, Jacob",686668,686608,NaN,called_strike,<NA>,<NA>,<NA>,<NA>,14,NaN,S,R,R,COL,KC,S,<NA>,NaN,0,0,2023,-0.34,1.04,0.993175,2.142476,669911,<NA>,666134,2,9,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,8.303227,-143.415483,-7.128019,-6.57895,32.571992,-16.371025,3.49,1.601,<NA>,<NA>,<NA>,99.2,2387,6.6,733583,682515,680769,665834,689374,686475,674646,683031,687614,53.88,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,82,1,4-Seam Fastball,5,8,5,8,8,5,5,8,NaN,NaN,215,-0.002,-0.052,<NA>,<NA>,<NA>,<NA>,0.052,<NA>,-3,-3,0.047,0.047,24,25,25,25,1,4,<NA>,<NA>,<NA>,<NA>,1.31,0.34,0.34,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,SI,2023-03-18,94.1,-2.72,4.62,"Snider, Collin",666134,676092,single,hit_into_play,<NA>,<NA>,<NA>,<NA>,14,Nolan Jones singles on a ground ball to center...,S,L,R,COL,KC,X,8,ground_ball,2,1,2023,-1.24,-0.05,0.39508,1.324593,663796,641385,669911,2,9,Bot,141.43,94.72,<NA>,<NA>,<NA>,<NA>,10.715535,-136.63842,-2.691955,-17.862125,29.758303,-32.37739,3.69,1.73,50,106.2,1,93.3,2522,5.9,733583,682515,680769,665834,689374,686475,674646,683031,687614,54.6,<NA>,<NA>,0.9,<NA>,1,0,4,81,4,Sinker,3,8,3,8,8,5,

In [ ]:
def coerce_numeric_cols(df: pd.DataFrame, skip: set[str] | None = None) -> pd.DataFrame:
    out = df.copy()
    skip = skip or set()
    for c in out.columns:
        if c in skip:
            continue
        if out[c].dtype == object:
            # Convert strings like '.321' or '1.234' when possible.
            converted = pd.to_numeric(out[c].astype(str).str.replace("%", "", regex=False), errors="ignore")
            out[c] = converted
    return out


def add_basic_game_outcome_features(games_df: pd.DataFrame) -> pd.DataFrame:
    g = games_df.copy()
    g["home_win"] = np.where(g["is_final"], (g["home_score"] > g["away_score"]).astype(float), np.nan)
    g["away_win"] = np.where(g["is_final"], (g["away_score"] > g["home_score"]).astype(float), np.nan)
    g["home_run_diff"] = np.where(g["is_final"], g["home_score"] - g["away_score"], np.nan)
    g["away_run_diff"] = np.where(g["is_final"], g["away_score"] - g["home_score"], np.nan)
    return g


def team_game_long_from_games(games_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, r in games_df.iterrows():
        if not bool(r.get("is_final")):
            continue
        for side in ["home", "away"]:
            opp = "away" if side == "home" else "home"
            runs_for = r.get(f"{side}_score")
            runs_against = r.get(f"{opp}_score")
            rows.append({
                "game_pk": r.get("game_pk"),
                "official_date": r.get("official_date"),
                "game_datetime_utc": r.get("game_datetime_utc"),
                "team_side": side,
                "team_id": r.get(f"{side}_team_id"),
                "team_name": r.get(f"{side}_team_name"),
                "team_norm": r.get(f"{side}_team_norm"),
                "opponent_team_id": r.get(f"{opp}_team_id"),
                "opponent_team_name": r.get(f"{opp}_team_name"),
                "runs_for": runs_for,
                "runs_against": runs_against,
                "win": float(runs_for > runs_against) if pd.notna(runs_for) and pd.notna(runs_against) else np.nan,
                "run_diff": runs_for - runs_against if pd.notna(runs_for) and pd.notna(runs_against) else np.nan,
            })
    return pd.DataFrame(rows)


def add_rolling_entity_features(long_df: pd.DataFrame, entity_col: str, value_cols: list[str], prefix: str,
                                windows: list[int] = [3, 5, 10, 20], season: bool = True) -> pd.DataFrame:
    if long_df.empty:
        return pd.DataFrame()
    d = long_df.copy().sort_values([entity_col, "official_date", "game_datetime_utc", "game_pk"])
    out = d[["game_pk", entity_col]].copy()
    for col in value_cols:
        d[col] = pd.to_numeric(d[col], errors="coerce")
        shifted = d.groupby(entity_col)[col].shift(1)
        if season:
            out[f"{prefix}_{col}_season_to_date"] = shifted.groupby(d[entity_col]).expanding(min_periods=1).mean().reset_index(level=0, drop=True)
        for w in windows:
            out[f"{prefix}_{col}_last{w}"] = shifted.groupby(d[entity_col]).rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
    return pd.concat([d[["game_pk", entity_col, "official_date", "game_datetime_utc"]].reset_index(drop=True), out.drop(columns=["game_pk", entity_col]).reset_index(drop=True)], axis=1)


def compute_elo_features(games_df: pd.DataFrame, k: float = 20.0, home_adv: float = 35.0, base_elo: float = 1500.0) -> pd.DataFrame:
    g = games_df.copy().sort_values(["official_date", "game_datetime_utc", "game_pk"])
    ratings: dict[int, float] = {}
    rows = []
    for _, r in g.iterrows():
        h = int(r["home_team_id"]) if pd.notna(r.get("home_team_id")) else None
        a = int(r["away_team_id"]) if pd.notna(r.get("away_team_id")) else None
        if h is None or a is None:
            continue
        rh = ratings.get(h, base_elo)
        ra = ratings.get(a, base_elo)
        ph = 1.0 / (1.0 + 10 ** (-((rh + home_adv) - ra) / 400.0))
        rows.append({
            "game_pk": r.get("game_pk"),
            "home_elo_pre": rh,
            "away_elo_pre": ra,
            "diff_elo_pre": rh - ra,
            "elo_home_win_prob": ph,
        })
        if bool(r.get("is_final")) and pd.notna(r.get("home_score")) and pd.notna(r.get("away_score")):
            outcome = 1.0 if r["home_score"] > r["away_score"] else 0.0
            change = k * (outcome - ph)
            ratings[h] = rh + change
            ratings[a] = ra - change
    return pd.DataFrame(rows)

print("Basic rolling/ELO functions ready")


Basic rolling/ELO functions ready


In [ ]:
def prepare_statcast_pitch_level(sc: pd.DataFrame) -> pd.DataFrame:
    if sc.empty:
        return pd.DataFrame()
    d = sc.copy()
    # Normalize important columns.
    rename_map = {"game_date": "official_date"}
    d = d.rename(columns={k: v for k, v in rename_map.items() if k in d.columns})
    if "game_pk" not in d.columns and "game_pk" in d.columns:
        pass
    if "official_date" in d.columns:
        d["official_date"] = pd.to_datetime(d["official_date"], errors="coerce")
    if "game_pk" in d.columns:
        d["game_pk"] = pd.to_numeric(d["game_pk"], errors="coerce")
    for c in ["release_speed", "release_spin_rate", "release_extension", "launch_speed", "launch_angle", "estimated_woba_using_speedangle", "woba_value", "estimated_ba_using_speedangle"]:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    # batting team and pitching team from inning half.
    if {"inning_topbot", "home_team", "away_team"}.issubset(d.columns):
        d["bat_team"] = np.where(d["inning_topbot"].astype(str).str.lower().eq("top"), d["away_team"], d["home_team"])
        d["pitch_team"] = np.where(d["inning_topbot"].astype(str).str.lower().eq("top"), d["home_team"], d["away_team"])
    d["bat_team_norm"] = d.get("bat_team", pd.Series(index=d.index, dtype=object)).apply(normalize_team_name)
    d["pitch_team_norm"] = d.get("pitch_team", pd.Series(index=d.index, dtype=object)).apply(normalize_team_name)

    # Helpful indicators.
    desc = d.get("description", pd.Series("", index=d.index)).fillna("").astype(str)
    events = d.get("events", pd.Series("", index=d.index)).fillna("").astype(str)
    d["is_pa_event"] = events.ne("")
    d["is_strikeout"] = events.str.contains("strikeout", case=False, na=False)
    d["is_walk"] = events.str.contains("walk", case=False, na=False) & ~events.str.contains("intent", case=False, na=False)
    d["is_home_run"] = events.str.contains("home_run", case=False, na=False)
    d["is_batted_ball"] = d.get("launch_speed", pd.Series(np.nan, index=d.index)).notna()
    d["is_hard_hit"] = d.get("launch_speed", pd.Series(np.nan, index=d.index)).ge(95)
    d["is_sweetspot"] = d.get("launch_angle", pd.Series(np.nan, index=d.index)).between(8, 32)
    d["is_whiff"] = desc.isin(["swinging_strike", "swinging_strike_blocked", "foul_tip"])
    d["is_called_strike"] = desc.eq("called_strike")
    d["is_swing"] = desc.str.contains("swing|foul|hit_into_play", case=False, regex=True, na=False)
    d["pitch_family"] = d.get("pitch_type", pd.Series("UNK", index=d.index)).fillna("UNK").map(pitch_family)
    return d


def pitch_family(pt: Any) -> str:
    if pt is None or pd.isna(pt):
        return "unknown"
    pt = str(pt).upper()
    fast = {"FF", "SI", "FC", "FA", "FS"}
    breaking = {"SL", "CU", "KC", "SV", "ST"}
    offspeed = {"CH", "FS", "FO", "SC"}
    if pt in fast:
        return "fastball"
    if pt in breaking:
        return "breaking"
    if pt in offspeed:
        return "offspeed"
    return "other"


def entropy_from_counts(counts: pd.Series) -> float:
    values = counts[counts > 0].astype(float)
    if values.sum() <= 0:
        return np.nan
    p = values / values.sum()
    return float(-(p * np.log(p)).sum())


def aggregate_statcast_team_game(sc: pd.DataFrame) -> pd.DataFrame:
    if sc.empty:
        return pd.DataFrame()
    d = prepare_statcast_pitch_level(sc)
    group_cols = ["game_pk", "official_date", "bat_team_norm"]
    agg = d.groupby(group_cols).agg(
        sc_pitches_seen=("game_pk", "size"),
        sc_pa=("is_pa_event", "sum"),
        sc_avg_ev=("launch_speed", "mean"),
        sc_max_ev=("launch_speed", "max"),
        sc_avg_la=("launch_angle", "mean"),
        sc_hard_hit_rate=("is_hard_hit", "mean"),
        sc_sweetspot_rate=("is_sweetspot", "mean"),
        sc_xwoba_contact=("estimated_woba_using_speedangle", "mean"),
        sc_woba=("woba_value", "mean"),
        sc_k_rate=("is_strikeout", "mean"),
        sc_bb_rate=("is_walk", "mean"),
        sc_hr_rate=("is_home_run", "mean"),
        sc_whiff_rate=("is_whiff", "mean"),
        sc_csw_rate=("is_called_strike", "mean"),
    ).reset_index().rename(columns={"bat_team_norm": "team_norm"})
    # CSW should include called strike + whiff over pitches.
    csw = d.assign(csw=d["is_called_strike"] | d["is_whiff"]).groupby(group_cols)["csw"].mean().reset_index(name="sc_csw_rate")
    agg = agg.drop(columns=["sc_csw_rate"], errors="ignore").merge(csw.rename(columns={"bat_team_norm": "team_norm"}), on=["game_pk", "official_date", "team_norm"], how="left")
    return agg


def aggregate_statcast_pitcher_game(sc: pd.DataFrame) -> pd.DataFrame:
    if sc.empty:
        return pd.DataFrame()
    d = prepare_statcast_pitch_level(sc)
    if "pitcher" not in d.columns:
        return pd.DataFrame()
    group_cols = ["game_pk", "official_date", "pitcher"]
    agg = d.groupby(group_cols).agg(
        sc_pitches=("game_pk", "size"),
        sc_pa=("is_pa_event", "sum"),
        sc_release_speed_mean=("release_speed", "mean"),
        sc_release_spin_mean=("release_spin_rate", "mean"),
        sc_release_extension_mean=("release_extension", "mean"),
        sc_avg_ev_allowed=("launch_speed", "mean"),
        sc_max_ev_allowed=("launch_speed", "max"),
        sc_avg_la_allowed=("launch_angle", "mean"),
        sc_hard_hit_rate_allowed=("is_hard_hit", "mean"),
        sc_sweetspot_rate_allowed=("is_sweetspot", "mean"),
        sc_xwoba_allowed_contact=("estimated_woba_using_speedangle", "mean"),
        sc_woba_allowed=("woba_value", "mean"),
        sc_k_rate=("is_strikeout", "mean"),
        sc_bb_rate=("is_walk", "mean"),
        sc_hr_rate=("is_home_run", "mean"),
        sc_whiff_rate=("is_whiff", "mean"),
    ).reset_index().rename(columns={"pitcher": "pitcher_id"})
    # pitch mix entropy by pitcher-game.
    mix = d.groupby(group_cols + ["pitch_family"]).size().rename("n").reset_index()
    ent = mix.groupby(group_cols)["n"].apply(entropy_from_counts).reset_index(name="sc_pitch_mix_entropy")
    ent = ent.rename(columns={"pitcher": "pitcher_id"})
    agg = agg.merge(ent, on=["game_pk", "official_date", "pitcher_id"], how="left")
    return agg


def aggregate_statcast_pitch_type(sc: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    if sc.empty:
        return pd.DataFrame(), pd.DataFrame()
    d = prepare_statcast_pitch_level(sc)
    team_pt = d.groupby(["game_pk", "official_date", "bat_team_norm", "pitch_family"]).agg(
        pitches=("game_pk", "size"),
        avg_ev=("launch_speed", "mean"),
        woba=("woba_value", "mean"),
        whiff_rate=("is_whiff", "mean"),
        hard_hit_rate=("is_hard_hit", "mean"),
    ).reset_index().rename(columns={"bat_team_norm": "team_norm"})
    pit_pt = d.groupby(["game_pk", "official_date", "pitcher", "pitch_family"]).agg(
        pitches=("game_pk", "size"),
        release_speed=("release_speed", "mean"),
        avg_ev_allowed=("launch_speed", "mean"),
        woba_allowed=("woba_value", "mean"),
        whiff_rate=("is_whiff", "mean"),
        hard_hit_rate_allowed=("is_hard_hit", "mean"),
    ).reset_index().rename(columns={"pitcher": "pitcher_id"})
    return team_pt, pit_pt

print("Statcast feature functions ready")


Statcast feature functions ready


In [ ]:
def rolling_team_features_from_games(games_df: pd.DataFrame) -> pd.DataFrame:
    long = team_game_long_from_games(games_df)
    if long.empty:
        return pd.DataFrame()
    value_cols = ["runs_for", "runs_against", "win", "run_diff"]
    return add_rolling_entity_features(long, "team_id", value_cols, "team")


def rolling_boxscore_features(box_df: pd.DataFrame) -> pd.DataFrame:
    if box_df.empty:
        return pd.DataFrame()
    d = coerce_numeric_cols(box_df, skip={"team_name", "team_norm", "opponent_team_name", "team_side"})
    value_cols = []
    for c in d.columns:
        if c.startswith("box_bat_") or c.startswith("box_pitch_") or c in ["runs_for", "runs_against"]:
            if pd.api.types.is_numeric_dtype(d[c]):
                value_cols.append(c)
    value_cols = value_cols[:80]  # keep lab manageable; remove cap if desired.
    return add_rolling_entity_features(d, "team_id", value_cols, "box")


def rolling_statcast_team_features(sc_team_game: pd.DataFrame) -> pd.DataFrame:
    if sc_team_game.empty:
        return pd.DataFrame()
    value_cols = [c for c in sc_team_game.columns if c.startswith("sc_")]
    return add_rolling_entity_features(sc_team_game, "team_norm", value_cols, "team_off")


def rolling_statcast_pitcher_features(sc_pitcher_game: pd.DataFrame) -> pd.DataFrame:
    if sc_pitcher_game.empty:
        return pd.DataFrame()
    value_cols = [c for c in sc_pitcher_game.columns if c.startswith("sc_")]
    return add_rolling_entity_features(sc_pitcher_game, "pitcher_id", value_cols, "starter_statcast")


def rolling_pitchmix_team_features(team_pt: pd.DataFrame) -> pd.DataFrame:
    if team_pt.empty:
        return pd.DataFrame()
    # Pivot pitch families wide per game/team.
    piv = team_pt.pivot_table(index=["game_pk", "official_date", "team_norm"], columns="pitch_family", values=["pitches", "avg_ev", "woba", "whiff_rate", "hard_hit_rate"], aggfunc="mean")
    piv.columns = [f"pt_{a}_{b}" for a, b in piv.columns]
    piv = piv.reset_index()
    value_cols = [c for c in piv.columns if c.startswith("pt_")]
    return add_rolling_entity_features(piv, "team_norm", value_cols, "team_pitchmix")

print("Rolling feature functions ready")


Rolling feature functions ready


In [ ]:
def merge_home_away_team_features(base: pd.DataFrame, feat: pd.DataFrame, entity_col: str, prefix: str, id_home_col: str, id_away_col: str) -> pd.DataFrame:
    if feat.empty:
        return base
    d = base.copy()
    feature_cols = [c for c in feat.columns if c not in {"game_pk", entity_col, "official_date", "game_datetime_utc"}]
    latest = feat[["game_pk", entity_col] + feature_cols].drop_duplicates(["game_pk", entity_col], keep="last")

    home = latest.rename(columns={entity_col: id_home_col, **{c: f"home_{prefix}_{c}" for c in feature_cols}})
    away = latest.rename(columns={entity_col: id_away_col, **{c: f"away_{prefix}_{c}" for c in feature_cols}})

    d = d.merge(home, on=["game_pk", id_home_col], how="left")
    d = d.merge(away, on=["game_pk", id_away_col], how="left")

    for c in feature_cols:
        hc = f"home_{prefix}_{c}"
        ac = f"away_{prefix}_{c}"
        if hc in d.columns and ac in d.columns:
            d[f"diff_{prefix}_{c}"] = d[hc] - d[ac]
    return d


def attach_latest_h2h_odds_features(games_df: pd.DataFrame, odds_df: pd.DataFrame) -> pd.DataFrame:
    d = games_df.copy()
    if odds_df.empty:
        return d
    h2h = odds_df[odds_df["market_key"].astype(str).str.lower().isin(["h2h", "moneyline"])]
    if h2h.empty:
        return d
    event_level = []
    for event_id, ev in h2h.groupby("event_id"):
        ev0 = ev.iloc[0]
        home_norm = ev0["home_team_norm"]
        away_norm = ev0["away_team_norm"]
        home_prices = pd.to_numeric(ev.loc[ev["outcome_name_norm"].eq(home_norm), "outcome_price"], errors="coerce").dropna()
        away_prices = pd.to_numeric(ev.loc[ev["outcome_name_norm"].eq(away_norm), "outcome_price"], errors="coerce").dropna()
        if home_prices.empty or away_prices.empty:
            continue
        hp = float(home_prices.median())
        ap = float(away_prices.median())
        home_nv, away_nv = no_vig_two_way_prob(hp, ap)
        event_level.append({
            "event_id": event_id,
            "odds_commence_time_utc": ev0["commence_time_utc"],
            "home_team_norm": home_norm,
            "away_team_norm": away_norm,
            "home_moneyline_median": hp,
            "away_moneyline_median": ap,
            "market_home_no_vig_prob": home_nv,
            "market_away_no_vig_prob": away_nv,
        })
    evdf = pd.DataFrame(event_level)
    if evdf.empty:
        return d
    # Match by team names and nearest start time within 3 hours.
    rows = []
    for idx, g in d.iterrows():
        cand = evdf[(evdf["home_team_norm"].eq(g.get("home_team_norm"))) & (evdf["away_team_norm"].eq(g.get("away_team_norm")))].copy()
        if cand.empty:
            rows.append({})
            continue
        cand["dt_min"] = (cand["odds_commence_time_utc"] - g["game_datetime_utc"]).abs().dt.total_seconds() / 60.0
        cand = cand.sort_values("dt_min")
        best = cand.iloc[0]
        if best["dt_min"] <= 180:
            rows.append(best.drop(labels=["home_team_norm", "away_team_norm"]).to_dict())
        else:
            rows.append({})
    attach = pd.DataFrame(rows)
    d = pd.concat([d.reset_index(drop=True), attach.reset_index(drop=True)], axis=1)
    d["has_market_odds"] = d.get("home_moneyline_median", pd.Series(np.nan, index=d.index)).notna().astype(int)
    return d


def build_game_feature_frame(games_df: pd.DataFrame, box_df: pd.DataFrame, statcast_raw_df: pd.DataFrame, odds_df: pd.DataFrame) -> pd.DataFrame:
    base = add_basic_game_outcome_features(games_df).copy()
    elo = compute_elo_features(base)
    if not elo.empty:
        base = base.merge(elo, on="game_pk", how="left")

    team_roll = rolling_team_features_from_games(base)
    base = merge_home_away_team_features(base, team_roll, "team_id", "team", "home_team_id", "away_team_id")

    box_roll = rolling_boxscore_features(box_df)
    base = merge_home_away_team_features(base, box_roll, "team_id", "box", "home_team_id", "away_team_id")

    if not statcast_raw_df.empty:
        sc_team = aggregate_statcast_team_game(statcast_raw_df)
        sc_pitcher = aggregate_statcast_pitcher_game(statcast_raw_df)
        team_pt, pit_pt = aggregate_statcast_pitch_type(statcast_raw_df)
        sc_team_roll = rolling_statcast_team_features(sc_team)
        base = merge_home_away_team_features(base, sc_team_roll, "team_norm", "team_sc", "home_team_norm", "away_team_norm")
        pt_roll = rolling_pitchmix_team_features(team_pt)
        base = merge_home_away_team_features(base, pt_roll, "team_norm", "team_pitchmix", "home_team_norm", "away_team_norm")
        sp_roll = rolling_statcast_pitcher_features(sc_pitcher)
        base = merge_home_away_starter_features(base, sp_roll)
    else:
        sc_team = sc_pitcher = team_pt = pit_pt = pd.DataFrame()

    base = attach_latest_h2h_odds_features(base, odds_df)
    return base


def merge_home_away_starter_features(base: pd.DataFrame, sp_feat: pd.DataFrame) -> pd.DataFrame:
    if sp_feat.empty:
        return base
    d = base.copy()
    feature_cols = [c for c in sp_feat.columns if c not in {"game_pk", "pitcher_id", "official_date", "game_datetime_utc"}]
    latest = sp_feat[["game_pk", "pitcher_id"] + feature_cols].drop_duplicates(["game_pk", "pitcher_id"], keep="last")
    home = latest.rename(columns={"pitcher_id": "home_probable_pitcher_id", **{c: f"home_starter_{c}" for c in feature_cols}})
    away = latest.rename(columns={"pitcher_id": "away_probable_pitcher_id", **{c: f"away_starter_{c}" for c in feature_cols}})
    d = d.merge(home, on=["game_pk", "home_probable_pitcher_id"], how="left")
    d = d.merge(away, on=["game_pk", "away_probable_pitcher_id"], how="left")
    for c in feature_cols:
        hc = f"home_starter_{c}"
        ac = f"away_starter_{c}"
        if hc in d.columns and ac in d.columns:
            d[f"diff_starter_{c}"] = d[hc] - d[ac]
    return d

print("Game feature builder ready")


Game feature builder ready


In [ ]:
statcast_raw.head()

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,tfs_deprecated,tfs_zulu_deprecated,umpire,sv_id,vx0,vy0,vz0,ax,ay,az,sz_top,sz_bot,hit_distance_sc,launch_speed,launch_angle,effective_speed,release_spin_rate,release_extension,game_pk,fielder_2,fielder_3,fielder_4,fielder_5,fielder_6,fielder_7,fielder_8,fielder_9,release_pos_y,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,pitch_name,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,miss_distance,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,SL,2023-03-18,84.7,-2.16,5.5,"Wallace, Jacob",686668,686608,force_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,9,"Brenton Doyle grounds into a force out, third ...",S,R,R,COL,KC,X,5,ground_ball,1,1,2023,0.78,-0.18,0.300694,1.855843,669911,<NA>,666134,2,9,Bot,100.33,162.86,<NA>,<NA>,<NA>,<NA>,4.185992,-123.367264,-1.825071,7.102894,24.276703,-33.89723,3.49,1.601,80,73.8,8,84.7,2999,6.2,733583,682515,680769,665834,689374,686475,674646,683031,687614,54.25,<NA>,<NA>,0.0,<NA>,0,0,2,82,3,Slider,5,8,5,8,8,5,5,8,Standard,Standard,51,-0.046,-0.301,<NA>,<NA>,<NA>,<NA>,0.301,88.0,-3,-3,0.046,0.046,24,25,25,25,1,4,<NA>,<NA>,<NA>,<NA>,3.35,-0.78,-0.78,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,FC,2023-03-18,91.4,-2.03,5.62,"Wallace, Jacob",686668,686608,NaN,ball,<NA>,<NA>,<NA>,<NA>,14,NaN,S,R,R,COL,KC,B,<NA>,NaN,0,1,2023,0.46,0.5,2.466818,1.229679,669911,<NA>,666134,2,9,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,10.359331,-132.664357,-6.290476,3.355396,27.370295,-25.062908,3.49,1.601,<NA>,<NA>,<NA>,91.7,2797,6.5,733583,682515,680769,665834,689374,686475,674646,683031,687614,53.96,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,82,2,Cutter,5,8,5,8,8,5,5,8,Standard,Standard,146,0.001,0.039,<NA>,<NA>,<NA>,<NA>,-0.039,<NA>,-3,-3,0.045,0.045,24,25,25,25,1,4,<NA>,<NA>,<NA>,<NA>,2.23,-0.46,-0.46,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,FF,2023-03-18,98.7,-1.75,5.86,"Wallace, Jacob",686668,686608,NaN,called_strike,<NA>,<NA>,<NA>,<NA>,14,NaN,S,R,R,COL,KC,S,<NA>,NaN,0,0,2023,-0.34,1.04,0.993175,2.142476,669911,<NA>,666134,2,9,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,8.303227,-143.415483,-7.128019,-6.57895,32.571992,-16.371025,3.49,1.601,<NA>,<NA>,<NA>,99.2,2387,6.6,733583,682515,680769,665834,689374,686475,674646,683031,687614,53.88,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,82,1,4-Seam Fastball,5,8,5,8,8,5,5,8,NaN,NaN,215,-0.002,-0.052,<NA>,<NA>,<NA>,<NA>,0.052,<NA>,-3,-3,0.047,0.047,24,25,25,25,1,4,<NA>,<NA>,<NA>,<NA>,1.31,0.34,0.34,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,SI,2023-03-18,94.1,-2.72,4.62,"Snider, Collin",666134,676092,single,hit_into_play,<NA>,<NA>,<NA>,<NA>,14,Nolan Jones singles on a ground ball to center...,S,L,R,COL,KC,X,8,ground_ball,2,1,2023,-1.24,-0.05,0.39508,1.324593,663796,641385,669911,2,9,Bot,141.43,94.72,<NA>,<NA>,<NA>,<NA>,10.715535,-136.63842,-2.691955,-17.862125,29.758303,-32.37739,3.69,1.73,50,106.2,1,93.3,2522,5.9,733583,682515,680769,665834,689374,686475,674646,683031,687614,54.6,<NA>,<NA>,0.9,<NA>,1,0,4,81,4,Sinker,3,8,3,8,8,5,

In [ ]:
def repair_game_time_from_games(df: pd.DataFrame, games_df: pd.DataFrame, name: str) -> pd.DataFrame:
    if df is None or df.empty:
        print(f"{name}: empty")
        return pd.DataFrame()

    d = df.copy()
    lookup = games_df[["game_pk", "official_date", "game_datetime_utc"]].drop_duplicates("game_pk").copy()
    lookup["game_pk"] = pd.to_numeric(lookup["game_pk"], errors="coerce")
    lookup["official_date"] = pd.to_datetime(lookup["official_date"], errors="coerce")
    lookup["game_datetime_utc"] = pd.to_datetime(lookup["game_datetime_utc"], errors="coerce", utc=True)

    d["game_pk"] = pd.to_numeric(d["game_pk"], errors="coerce")

    if "official_date" not in d.columns and "game_date" in d.columns:
        d["official_date"] = pd.to_datetime(d["game_date"], errors="coerce")

    d = d.merge(
        lookup.rename(columns={
            "official_date": "_official_date_from_games",
            "game_datetime_utc": "_game_datetime_utc_from_games",
        }),
        on="game_pk",
        how="left",
    )

    if "official_date" in d.columns:
        d["official_date"] = pd.to_datetime(d["official_date"], errors="coerce")
        d["official_date"] = d["official_date"].combine_first(d["_official_date_from_games"])
    else:
        d["official_date"] = d["_official_date_from_games"]

    if "game_datetime_utc" in d.columns:
        d["game_datetime_utc"] = pd.to_datetime(d["game_datetime_utc"], errors="coerce", utc=True)
        d["game_datetime_utc"] = d["game_datetime_utc"].combine_first(d["_game_datetime_utc_from_games"])
    else:
        d["game_datetime_utc"] = d["_game_datetime_utc_from_games"]

    d = d.drop(columns=["_official_date_from_games", "_game_datetime_utc_from_games"], errors="ignore")

    print(
        f"{name}: repaired timestamps; missing game_datetime_utc = "
        f"{d['game_datetime_utc'].isna().sum():,} of {len(d):,}"
    )
    return d


statcast_raw = repair_game_time_from_games(statcast_raw, games, "statcast_raw")

print(statcast_raw[["game_pk", "official_date", "game_datetime_utc"]].head())
print(statcast_raw["game_datetime_utc"].isna().sum())

statcast_raw: repaired timestamps; missing game_datetime_utc = 197,968 of 2,645,941
   game_pk official_date game_datetime_utc
0   733583    2023-03-18               NaT
1   733583    2023-03-18               NaT
2   733583    2023-03-18               NaT
3   733583    2023-03-18               NaT
4   733583    2023-03-18               NaT
197968


In [ ]:
games_pks = set(pd.to_numeric(games["game_pk"], errors="coerce").dropna().astype(int))

before = len(statcast_raw)

statcast_raw["game_pk"] = pd.to_numeric(statcast_raw["game_pk"], errors="coerce")
statcast_raw = statcast_raw[statcast_raw["game_pk"].isin(games_pks)].copy()

# Re-run timestamp repair after filtering.
statcast_raw = repair_game_time_from_games(statcast_raw, games, "statcast_raw")

after = len(statcast_raw)

print(f"Statcast rows filtered to scheduled games: {before:,} -> {after:,}")
print("Missing game_datetime_utc:", statcast_raw["game_datetime_utc"].isna().sum())

assert statcast_raw["game_datetime_utc"].notna().all()

statcast_raw: repaired timestamps; missing game_datetime_utc = 0 of 2,447,973
Statcast rows filtered to scheduled games: 2,645,941 -> 2,447,973
Missing game_datetime_utc: 0


In [ ]:
def prepare_statcast_pitch_level(sc: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare raw pybaseball Statcast rows without creating duplicate official_date columns.

    repair_game_time_from_games() may already add official_date and game_datetime_utc
    while leaving pybaseball's game_date in place. Do NOT rename game_date -> official_date
    when official_date already exists.
    """
    if sc is None or sc.empty:
        return pd.DataFrame()

    d = sc.copy()

    # Defensive cleanup in case a prior notebook run created duplicate labels.
    if not d.columns.is_unique:
        out = pd.DataFrame(index=d.index)

        for col in pd.Index(d.columns).unique():
            same = d.loc[:, d.columns == col]

            if same.shape[1] == 1:
                out[col] = same.iloc[:, 0]
            else:
                s = same.iloc[:, 0]
                for j in range(1, same.shape[1]):
                    s = s.combine_first(same.iloc[:, j])
                out[col] = s

        d = out

    # Use official_date if present; otherwise derive it from pybaseball game_date.
    if "official_date" in d.columns:
        d["official_date"] = pd.to_datetime(d["official_date"], errors="coerce")
    elif "game_date" in d.columns:
        d["official_date"] = pd.to_datetime(d["game_date"], errors="coerce")

    # Keep UTC start time from MLB schedule lookup if available.
    if "game_datetime_utc" in d.columns:
        d["game_datetime_utc"] = pd.to_datetime(
            d["game_datetime_utc"],
            errors="coerce",
            utc=True,
        )
    elif "official_date" in d.columns:
        d["game_datetime_utc"] = pd.to_datetime(
            d["official_date"],
            errors="coerce",
            utc=True,
        )

    if "game_pk" in d.columns:
        d["game_pk"] = pd.to_numeric(d["game_pk"], errors="coerce")

    for c in [
        "release_speed",
        "release_spin_rate",
        "release_extension",
        "launch_speed",
        "launch_angle",
        "estimated_woba_using_speedangle",
        "woba_value",
        "estimated_ba_using_speedangle",
    ]:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    if {"inning_topbot", "home_team", "away_team"}.issubset(d.columns):
        d["bat_team"] = np.where(
            d["inning_topbot"].astype(str).str.lower().eq("top"),
            d["away_team"],
            d["home_team"],
        )

        d["pitch_team"] = np.where(
            d["inning_topbot"].astype(str).str.lower().eq("top"),
            d["home_team"],
            d["away_team"],
        )

    d["bat_team_norm"] = d.get(
        "bat_team",
        pd.Series(index=d.index, dtype=object),
    ).apply(normalize_team_name)

    d["pitch_team_norm"] = d.get(
        "pitch_team",
        pd.Series(index=d.index, dtype=object),
    ).apply(normalize_team_name)

    desc = d.get("description", pd.Series("", index=d.index)).fillna("").astype(str)
    events = d.get("events", pd.Series("", index=d.index)).fillna("").astype(str)

    d["is_pa_event"] = events.ne("")
    d["is_strikeout"] = events.str.contains("strikeout", case=False, na=False)
    d["is_walk"] = (
        events.str.contains("walk", case=False, na=False)
        & ~events.str.contains("intent", case=False, na=False)
    )
    d["is_home_run"] = events.str.contains("home_run", case=False, na=False)

    d["is_batted_ball"] = d.get(
        "launch_speed",
        pd.Series(np.nan, index=d.index),
    ).notna()

    d["is_hard_hit"] = d.get(
        "launch_speed",
        pd.Series(np.nan, index=d.index),
    ).ge(95)

    d["is_sweetspot"] = d.get(
        "launch_angle",
        pd.Series(np.nan, index=d.index),
    ).between(8, 32)

    d["is_whiff"] = desc.isin(
        ["swinging_strike", "swinging_strike_blocked", "foul_tip"]
    )
    d["is_called_strike"] = desc.eq("called_strike")
    d["is_swing"] = desc.str.contains(
        "swing|foul|hit_into_play",
        case=False,
        regex=True,
        na=False,
    )

    d["pitch_family"] = d.get(
        "pitch_type",
        pd.Series("UNK", index=d.index),
    ).fillna("UNK").map(pitch_family)

    return d

In [ ]:
# -----------------------------------------------------------------------------
# PATCH: preserve game_datetime_utc through Statcast aggregations
# -----------------------------------------------------------------------------
# The raw Statcast table has game_datetime_utc after repair, but the earlier
# aggregate_statcast_* functions grouped only by game_pk/official_date/team.
# That dropped game_datetime_utc, and the rolling feature function then failed
# with KeyError: 'game_datetime_utc'. These definitions override the prior ones.

def _ensure_game_time_columns(d: pd.DataFrame, name: str = "df") -> pd.DataFrame:
    if d is None or d.empty:
        return pd.DataFrame() if d is None else d.copy()
    out = d.copy()
    if not out.columns.is_unique:
        dedup = pd.DataFrame(index=out.index)
        for col in pd.Index(out.columns).unique():
            same = out.loc[:, out.columns == col]
            if same.shape[1] == 1:
                dedup[col] = same.iloc[:, 0]
            else:
                s = same.iloc[:, 0]
                for j in range(1, same.shape[1]):
                    s = s.combine_first(same.iloc[:, j])
                dedup[col] = s
        out = dedup

    if "official_date" in out.columns:
        out["official_date"] = pd.to_datetime(out["official_date"], errors="coerce")
    elif "game_date" in out.columns:
        out["official_date"] = pd.to_datetime(out["game_date"], errors="coerce")

    if "game_datetime_utc" in out.columns:
        out["game_datetime_utc"] = pd.to_datetime(out["game_datetime_utc"], errors="coerce", utc=True)
    elif "official_date" in out.columns:
        # Fallback only. It preserves chronological ordering by date when exact first pitch is unavailable.
        out["game_datetime_utc"] = pd.to_datetime(out["official_date"], errors="coerce", utc=True)
    else:
        raise KeyError(f"{name} is missing both game_datetime_utc and official_date. Columns: {list(out.columns)}")

    if "game_pk" in out.columns:
        out["game_pk"] = pd.to_numeric(out["game_pk"], errors="coerce")
    return out


def add_rolling_entity_features(long_df: pd.DataFrame, entity_col: str, value_cols: list[str], prefix: str,
                                windows: list[int] = [3, 5, 10, 20], season: bool = True) -> pd.DataFrame:
    if long_df is None or long_df.empty:
        return pd.DataFrame()
    d = _ensure_game_time_columns(long_df, f"rolling_input_{prefix}")
    if entity_col not in d.columns:
        raise KeyError(f"{prefix}: missing entity column {entity_col}. Columns: {list(d.columns)}")
    sort_cols = [entity_col, "official_date", "game_datetime_utc", "game_pk"]
    d = d.sort_values(sort_cols).reset_index(drop=True)

    out = d[["game_pk", entity_col]].copy()
    for col in value_cols:
        if col not in d.columns:
            continue
        d[col] = pd.to_numeric(d[col], errors="coerce")
        shifted = d.groupby(entity_col)[col].shift(1)
        if season:
            out[f"{prefix}_{col}_season_to_date"] = (
                shifted.groupby(d[entity_col])
                .expanding(min_periods=1)
                .mean()
                .reset_index(level=0, drop=True)
            )
        for w in windows:
            out[f"{prefix}_{col}_last{w}"] = (
                shifted.groupby(d[entity_col])
                .rolling(w, min_periods=1)
                .mean()
                .reset_index(level=0, drop=True)
            )

    return pd.concat(
        [
            d[["game_pk", entity_col, "official_date", "game_datetime_utc"]].reset_index(drop=True),
            out.drop(columns=["game_pk", entity_col], errors="ignore").reset_index(drop=True),
        ],
        axis=1,
    )


def aggregate_statcast_team_game(sc: pd.DataFrame) -> pd.DataFrame:
    if sc is None or sc.empty:
        return pd.DataFrame()
    d = prepare_statcast_pitch_level(sc)
    d = _ensure_game_time_columns(d, "prepared_statcast_team")
    group_cols = ["game_pk", "official_date", "game_datetime_utc", "bat_team_norm"]
    agg = d.groupby(group_cols).agg(
        sc_pitches_seen=("game_pk", "size"),
        sc_pa=("is_pa_event", "sum"),
        sc_avg_ev=("launch_speed", "mean"),
        sc_max_ev=("launch_speed", "max"),
        sc_avg_la=("launch_angle", "mean"),
        sc_hard_hit_rate=("is_hard_hit", "mean"),
        sc_sweetspot_rate=("is_sweetspot", "mean"),
        sc_xwoba_contact=("estimated_woba_using_speedangle", "mean"),
        sc_woba=("woba_value", "mean"),
        sc_k_rate=("is_strikeout", "mean"),
        sc_bb_rate=("is_walk", "mean"),
        sc_hr_rate=("is_home_run", "mean"),
        sc_whiff_rate=("is_whiff", "mean"),
        sc_swing_rate=("is_swing", "mean"),
    ).reset_index().rename(columns={"bat_team_norm": "team_norm"})

    csw = (
        d.assign(csw=d["is_called_strike"] | d["is_whiff"])
        .groupby(group_cols)["csw"]
        .mean()
        .reset_index(name="sc_csw_rate")
        .rename(columns={"bat_team_norm": "team_norm"})
    )
    agg = agg.merge(csw, on=["game_pk", "official_date", "game_datetime_utc", "team_norm"], how="left")
    return agg


def aggregate_statcast_pitcher_game(sc: pd.DataFrame) -> pd.DataFrame:
    if sc is None or sc.empty:
        return pd.DataFrame()
    d = prepare_statcast_pitch_level(sc)
    d = _ensure_game_time_columns(d, "prepared_statcast_pitcher")
    if "pitcher" not in d.columns:
        return pd.DataFrame()
    group_cols = ["game_pk", "official_date", "game_datetime_utc", "pitcher"]
    agg = d.groupby(group_cols).agg(
        sc_pitches=("game_pk", "size"),
        sc_avg_velo=("release_speed", "mean"),
        sc_max_velo=("release_speed", "max"),
        sc_avg_spin=("release_spin_rate", "mean"),
        sc_avg_extension=("release_extension", "mean"),
        sc_whiff_rate=("is_whiff", "mean"),
        sc_called_strike_rate=("is_called_strike", "mean"),
        sc_swing_rate=("is_swing", "mean"),
        sc_hard_hit_allowed=("is_hard_hit", "mean"),
        sc_xwoba_allowed=("estimated_woba_using_speedangle", "mean"),
        sc_woba_allowed=("woba_value", "mean"),
        sc_k_rate=("is_strikeout", "mean"),
        sc_bb_rate=("is_walk", "mean"),
        sc_hr_rate=("is_home_run", "mean"),
    ).reset_index().rename(columns={"pitcher": "pitcher_id"})

    ent = (
        d.groupby(group_cols + ["pitch_family"]).size()
        .reset_index(name="n")
        .groupby(group_cols)["n"]
        .apply(entropy_from_counts)
        .reset_index(name="sc_pitchmix_entropy")
        .rename(columns={"pitcher": "pitcher_id"})
    )
    agg = agg.merge(ent, on=["game_pk", "official_date", "game_datetime_utc", "pitcher_id"], how="left")
    return agg


def aggregate_statcast_pitch_type(sc: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    if sc is None or sc.empty:
        return pd.DataFrame(), pd.DataFrame()
    d = prepare_statcast_pitch_level(sc)
    d = _ensure_game_time_columns(d, "prepared_statcast_pitch_type")

    team_pt = d.groupby(["game_pk", "official_date", "game_datetime_utc", "bat_team_norm", "pitch_family"]).agg(
        pitches=("game_pk", "size"),
        avg_ev=("launch_speed", "mean"),
        woba=("woba_value", "mean"),
        whiff_rate=("is_whiff", "mean"),
        hard_hit_rate=("is_hard_hit", "mean"),
    ).reset_index().rename(columns={"bat_team_norm": "team_norm"})

    if "pitcher" in d.columns:
        pit_pt = d.groupby(["game_pk", "official_date", "game_datetime_utc", "pitcher", "pitch_family"]).agg(
            pitches=("game_pk", "size"),
            avg_velo=("release_speed", "mean"),
            whiff_rate=("is_whiff", "mean"),
            usage=("game_pk", "size"),
        ).reset_index().rename(columns={"pitcher": "pitcher_id"})
    else:
        pit_pt = pd.DataFrame()
    return team_pt, pit_pt


def rolling_pitchmix_team_features(team_pt: pd.DataFrame) -> pd.DataFrame:
    if team_pt is None or team_pt.empty:
        return pd.DataFrame()
    d = _ensure_game_time_columns(team_pt, "team_pitchmix")
    piv = d.pivot_table(
        index=["game_pk", "official_date", "game_datetime_utc", "team_norm"],
        columns="pitch_family",
        values=["pitches", "avg_ev", "woba", "whiff_rate", "hard_hit_rate"],
        aggfunc="mean",
    )
    piv.columns = [f"pt_{a}_{b}" for a, b in piv.columns]
    piv = piv.reset_index()
    value_cols = [c for c in piv.columns if c.startswith("pt_")]
    return add_rolling_entity_features(piv, "team_norm", value_cols, "team_pitchmix")

print("Patched Statcast aggregation/rolling functions to preserve game_datetime_utc")


Patched Statcast aggregation/rolling functions to preserve game_datetime_utc


In [ ]:
def normalize_team_name(x):
    if x is None or pd.isna(x):
        return ""

    s = str(x).strip().lower()

    # Basic cleanup
    s = s.replace(".", "")
    s = s.replace("'", "")
    s = s.replace("&", "and")
    s = " ".join(s.split())

    aliases = {
        # Full-name cleanup
        "arizona dbacks": "arizona diamondbacks",
        "diamondbacks": "arizona diamondbacks",
        "dbacks": "arizona diamondbacks",
        "d-backs": "arizona diamondbacks",

        "white sox": "chicago white sox",
        "red sox": "boston red sox",
        "la dodgers": "los angeles dodgers",
        "la angels": "los angeles angels",
        "st louis cardinals": "st louis cardinals",
        "st. louis cardinals": "st louis cardinals",
        "oakland athletics": "athletics",
        "athletics": "athletics",

        # Statcast / MLB abbreviations
        "ari": "arizona diamondbacks",
        "az": "arizona diamondbacks",
        "atl": "atlanta braves",
        "bal": "baltimore orioles",
        "bos": "boston red sox",
        "chc": "chicago cubs",
        "chw": "chicago white sox",
        "cws": "chicago white sox",
        "cin": "cincinnati reds",
        "cle": "cleveland guardians",
        "col": "colorado rockies",
        "det": "detroit tigers",
        "hou": "houston astros",
        "kc": "kansas city royals",
        "kcr": "kansas city royals",
        "laa": "los angeles angels",
        "ana": "los angeles angels",
        "lad": "los angeles dodgers",
        "mia": "miami marlins",
        "mil": "milwaukee brewers",
        "min": "minnesota twins",
        "nym": "new york mets",
        "nyy": "new york yankees",
        "oak": "athletics",
        "ath": "athletics",
        "phi": "philadelphia phillies",
        "pit": "pittsburgh pirates",
        "sd": "san diego padres",
        "sdp": "san diego padres",
        "sf": "san francisco giants",
        "sfg": "san francisco giants",
        "sea": "seattle mariners",
        "stl": "st louis cardinals",
        "tb": "tampa bay rays",
        "tbr": "tampa bay rays",
        "tex": "texas rangers",
        "tor": "toronto blue jays",
        "wsh": "washington nationals",
        "was": "washington nationals",
        "wsn": "washington nationals",
    }

    return aliases.get(s, s)

In [ ]:
games = games.copy()

games["home_team_norm"] = games["home_team_name"].apply(normalize_team_name)
games["away_team_norm"] = games["away_team_name"].apply(normalize_team_name)

# Recompute Statcast batting/pitching team keys.
if {"inning_topbot", "home_team", "away_team"}.issubset(statcast_raw.columns):
    statcast_raw = statcast_raw.copy()

    statcast_raw["bat_team"] = np.where(
        statcast_raw["inning_topbot"].astype(str).str.lower().eq("top"),
        statcast_raw["away_team"],
        statcast_raw["home_team"],
    )

    statcast_raw["pitch_team"] = np.where(
        statcast_raw["inning_topbot"].astype(str).str.lower().eq("top"),
        statcast_raw["home_team"],
        statcast_raw["away_team"],
    )

    statcast_raw["bat_team_norm"] = statcast_raw["bat_team"].apply(normalize_team_name)
    statcast_raw["pitch_team_norm"] = statcast_raw["pitch_team"].apply(normalize_team_name)

In [ ]:
sc_team_keys_raw = sorted(
    {
        normalize_team_name(x)
        for x in pd.concat(
            [
                statcast_raw.get("home_team", pd.Series(dtype=object)),
                statcast_raw.get("away_team", pd.Series(dtype=object)),
            ],
            ignore_index=True,
        ).dropna().unique()
    }
)

game_team_keys = sorted(
    set(games["home_team_norm"].dropna().astype(str))
    | set(games["away_team_norm"].dropna().astype(str))
)

print("Statcast keys not in games:")
print(sorted(set(sc_team_keys_raw) - set(game_team_keys)))

print("\nGame keys not in Statcast:")
print(sorted(set(game_team_keys) - set(sc_team_keys_raw)))

Statcast keys not in games:
[]

Game keys not in Statcast:
[]


In [ ]:
features = build_game_feature_frame(games, box_team_game, statcast_raw, odds_snapshots)
print("features", features.shape)
print("date range", features["official_date"].min(), features["official_date"].max())
print("completed rows", features["target_home_win"].notna().sum())
display(features.head())


features (8365, 1965)
date range 2023-03-30 00:00:00 2026-09-22 00:00:00
completed rows 8313


,game_pk,official_date,game_datetime_utc,game_type,detailed_state,abstract_state,home_team_id,home_team_name,home_team_norm,away_team_id,away_team_name,away_team_norm,home_score,away_score,home_probable_pitcher_id,home_probable_pitcher_name,away_probable_pitcher_id,away_probable_pitcher_name,is_final,target_home_win,target_total_runs,target_home_margin,home_win,away_win,home_run_diff,away_run_diff,home_elo_pre,away_elo_pre,diff_elo_pre,elo_home_win_prob,home_team_team_runs_for_season_to_date,home_team_team_runs_for_last3,home_team_team_runs_for_last5,home_team_team_runs_for_last10,home_team_team_runs_for_last20,home_team_team_runs_against_season_to_date,home_team_team_runs_against_last3,home_team_team_runs_against_last5,home_team_team_runs_against_last10,home_team_team_runs_against_last20,home_team_team_win_season_to_date,home_team_team_win_last3,home_team_team_win_last5,home_team_team_win_last10,home_team_team_win_last20,home_team_team_run_diff_season_to_date,home_team_team_run_diff_last3,home_team_team_run_diff_last5,home_team_team_run_diff_last10,home_team_team_run_diff_last20,away_team_team_runs_for_season_to_date,away_team_team_runs_for_last3,away_team_team_runs_for_last5,away_team_team_runs_for_last10,away_team_team_runs_for_last20,away_team_team_runs_against_season_to_date,away_team_team_runs_against_last3,away_team_team_runs_against_last5,away_team_team_runs_against_last10,away_team_team_runs_against_last20,away_team_team_win_season_to_date,away_team_team_win_last3,away_team_team_win_last5,away_team_team_win_last10,away_team_team_win_last20,away_team_team_run_diff_season_to_date,away_team_team_run_diff_last3,away_team_team_run_diff_last5,away_team_team_run_diff_last10,away_team_team_run_diff_last20,diff_team_team_runs_for_season_to_date,diff_team_team_runs_for_last3,diff_team_team_runs_for_last5,diff_team_team_runs_for_last10,diff_team_team_runs_for_last20,diff_team_team_runs_against_season_to_date,diff_team_team_runs_against_last3,diff_team_team_runs_against_last5,diff_team_team_runs_against_last10,diff_team_team_runs_against_last20,diff_team_team_win_season_to_date,diff_team_team_win_last3,diff_team_team_win_last5,diff_team_team_win_last10,diff_team_team_win_last20,diff_team_team_run_diff_season_to_date,diff_team_team_run_diff_last3,diff_team_team_run_diff_last5,diff_team_team_run_diff_last10,diff_team_team_run_diff_last20,home_box_box_runs_for_season_to_date,home_box_box_runs_for_last3,home_box_box_runs_for_last5,home_box_box_runs_for_last10,home_box_box_runs_for_last20,home_box_box_runs_against_season_to_date,home_box_box_runs_against_last3,home_box_box_runs_against_last5,home_box_box_runs_against_last10,home_box_box_runs_against_last20,home_box_box_box_bat_flyOuts_season_to_date,home_box_box_box_bat_flyOuts_last3,home_box_box_box_bat_flyOuts_last5,home_box_box_box_bat_flyOuts_last10,home_box_box_box_bat_flyOuts_last20,home_box_box_box_bat_groundOuts_season_to_date,home_box_box_box_bat_groundOuts_last3,home_box_box_box_bat_groundOuts_last5,home_box_box_box_bat_groundOuts_last10,home_box_box_box_bat_groundOuts_last20,home_box_box_box_bat_airOuts_season_to_date,home_box_box_box_bat_airOuts_last3,home_box_box_box_bat_airOuts_last5,home_box_box_box_bat_airOuts_last10,home_box_box_box_bat_airOuts_last20,home_box_box_box_bat_runs_season_to_date,home_box_box_box_bat_runs_last3,home_box_box_box_bat_runs_last5,home_box_box_box_bat_runs_last10,home_box_box_box_bat_runs_last20,home_box_box_box_bat_doubles_season_to_date,home_box_box_box_bat_doubles_last3,home_box_box_box_bat_doubles_last5,home_box_box_box_bat_doubles_last10,home_box_box_box_bat_doubles_last20,...,away_starter_starter_statcast_sc_whiff_rate_season_to_date,away_starter_starter_statcast_sc_whiff_rate_last3,away_starter_starter_statcast_sc_whiff_rate_last5,away_starter_starter_statcast_sc_whiff_rate_last10,away_starter_starter_statcast_sc_whiff_rate_last20,away_starter_starter_statcast_sc_called_strike_rate_season_to_date,away_starter_starter_statcast_sc_called_s

In [ ]:
# Example custom interaction features. Add your experiments here.
features = features.copy()

if {"elo_home_win_prob", "market_home_no_vig_prob"}.issubset(features.columns):
    features["diff_elo_minus_market_home_prob"] = features["elo_home_win_prob"] - features["market_home_no_vig_prob"]

# Example: absolute model-independent favorite strength proxies.
if "diff_elo_pre" in features.columns:
    features["abs_diff_elo_pre"] = features["diff_elo_pre"].abs()

print("features after custom sandbox", features.shape)


features after custom sandbox (8365, 1966)


In [ ]:
print('Length of dataset compared to Features is ', len(features), ' compared to: ', len(features.columns))

Length of dataset compared to Features is  8365  compared to:  1966


In [ ]:
features.select_dtypes('float64','int64')

,home_score,away_score,home_probable_pitcher_id,away_probable_pitcher_id,target_home_win,target_total_runs,target_home_margin,home_win,away_win,home_run_diff,away_run_diff,home_elo_pre,away_elo_pre,diff_elo_pre,elo_home_win_prob,home_team_team_runs_for_season_to_date,home_team_team_runs_for_last3,home_team_team_runs_for_last5,home_team_team_runs_for_last10,home_team_team_runs_for_last20,home_team_team_runs_against_season_to_date,home_team_team_runs_against_last3,home_team_team_runs_against_last5,home_team_team_runs_against_last10,home_team_team_runs_against_last20,home_team_team_win_season_to_date,home_team_team_win_last3,home_team_team_win_last5,home_team_team_win_last10,home_team_team_win_last20,home_team_team_run_diff_season_to_date,home_team_team_run_diff_last3,home_team_team_run_diff_last5,home_team_team_run_diff_last10,home_team_team_run_diff_last20,away_team_team_runs_for_season_to_date,away_team_team_runs_for_last3,away_team_team_runs_for_last5,away_team_team_runs_for_last10,away_team_team_runs_for_last20,away_team_team_runs_against_season_to_date,away_team_team_runs_against_last3,away_team_team_runs_against_last5,away_team_team_runs_against_last10,away_team_team_runs_against_last20,away_team_team_win_season_to_date,away_team_team_win_last3,away_team_team_win_last5,away_team_team_win_last10,away_team_team_win_last20,away_team_team_run_diff_season_to_date,away_team_team_run_diff_last3,away_team_team_run_diff_last5,away_team_team_run_diff_last10,away_team_team_run_diff_last20,diff_team_team_runs_for_season_to_date,diff_team_team_runs_for_last3,diff_team_team_runs_for_last5,diff_team_team_runs_for_last10,diff_team_team_runs_for_last20,diff_team_team_runs_against_season_to_date,diff_team_team_runs_against_last3,diff_team_team_runs_against_last5,diff_team_team_runs_against_last10,diff_team_team_runs_against_last20,diff_team_team_win_season_to_date,diff_team_team_win_last3,diff_team_team_win_last5,diff_team_team_win_last10,diff_team_team_win_last20,diff_team_team_run_diff_season_to_date,diff_team_team_run_diff_last3,diff_team_team_run_diff_last5,diff_team_team_run_diff_last10,diff_team_team_run_diff_last20,home_box_box_runs_for_season_to_date,home_box_box_runs_for_last3,home_box_box_runs_for_last5,home_box_box_runs_for_last10,home_box_box_runs_for_last20,home_box_box_runs_against_season_to_date,home_box_box_runs_against_last3,home_box_box_runs_against_last5,home_box_box_runs_against_last10,home_box_box_runs_against_last20,home_box_box_box_bat_flyOuts_season_to_date,home_box_box_box_bat_flyOuts_last3,home_box_box_box_bat_flyOuts_last5,home_box_box_box_bat_flyOuts_last10,home_box_box_box_bat_flyOuts_last20,home_box_box_box_bat_groundOuts_season_to_date,home_box_box_box_bat_groundOuts_last3,home_box_box_box_bat_groundOuts_last5,home_box_box_box_bat_groundOuts_last10,home_box_box_box_bat_groundOuts_last20,home_box_box_box_bat_airOuts_season_to_date,home_box_box_box_bat_airOuts_last3,home_box_box_box_bat_airOuts_last5,home_box_box_box_bat_airOuts_last10,home_box_box_box_bat_airOuts_last20,home_box_box_box_bat_runs_season_to_date,home_box_box_box_bat_runs_last3,home_box_box_box_bat_runs_last5,home_box_box_box_bat_runs_last10,home_box_box_box_bat_runs_last20,home_box_box_box_bat_doubles_season_to_date,home_box_box_box_bat_doubles_last3,home_box_box_box_bat_doubles_last5,home_box_box_box_bat_doubles_last10,home_box_box_box_bat_doubles_last20,home_box_box_box_bat_triples_season_to_date,home_box_box_box_bat_triples_last3,home_box_box_box_bat_triples_last5,home_box_box_box_bat_triples_last10,home_box_box_box_bat_triples_last20,home_box_box_box_bat_homeRuns_season_to_date,home_box_box_box_bat_homeRuns_last3,home_box_box_box_bat_homeRuns_last5,home_box_box_box_bat_homeRuns_last10,home_box_box_box_bat_homeRuns_last20,home_box_box_box_bat_strikeOuts_season_to_date,home_box_box_box_bat_strikeOuts_last3,home_box_box_box_bat_strikeOuts_last5,home_box_box_box_bat_strikeOuts_last10,home_box_box_box_bat_strikeOuts_last20,...,away_starter_starter

In [ ]:
null_cols = features.columns[(features.isnull().sum() > 0) & (features.isnull().sum() < .9)]
fully_null_cols = features.columns[(features.isnull().sum() / len(features) >= .9)]

print('Length of columns under 100% missing values: ', len(null_cols))
for x in features.columns[features.isnull().sum() > 0]:
  print(x, ' ', features[x].isnull().sum() / len(features))

for col in fully_null_cols:
  print('Column with 100% missing values: ', col)

Length of columns under 100% missing values:  0
home_score   0.006216377764494919
away_score   0.006216377764494919
home_probable_pitcher_id   0.0015540944411237298
home_probable_pitcher_name   0.0015540944411237298
away_probable_pitcher_id   0.0008368200836820083
away_probable_pitcher_name   0.0008368200836820083
target_home_win   0.006216377764494919
target_total_runs   0.006216377764494919
target_home_margin   0.006216377764494919
home_win   0.005379557680812911
away_win   0.005379557680812911
home_run_diff   0.006216377764494919
away_run_diff   0.006216377764494919
home_team_team_runs_for_season_to_date   0.007172743574417215
home_team_team_runs_for_last3   0.007172743574417215
home_team_team_runs_for_last5   0.007172743574417215
home_team_team_runs_for_last10   0.007172743574417215
home_team_team_runs_for_last20   0.007172743574417215
home_team_team_runs_against_season_to_date   0.007172743574417215
home_team_team_runs_against_last3   0.007172743574417215
home_team_team_runs_again

In [ ]:
df_na = features.drop(fully_null_cols, axis=1)

In [ ]:
for x in df_na.columns[df_na.isnull().sum() > .5]:
  print(x, ' ', df_na[x].isnull().sum() / len(df_na))

home_score   0.006216377764494919
away_score   0.006216377764494919
home_probable_pitcher_id   0.0015540944411237298
home_probable_pitcher_name   0.0015540944411237298
away_probable_pitcher_id   0.0008368200836820083
away_probable_pitcher_name   0.0008368200836820083
target_home_win   0.006216377764494919
target_total_runs   0.006216377764494919
target_home_margin   0.006216377764494919
home_win   0.005379557680812911
away_win   0.005379557680812911
home_run_diff   0.006216377764494919
away_run_diff   0.006216377764494919
home_team_team_runs_for_season_to_date   0.007172743574417215
home_team_team_runs_for_last3   0.007172743574417215
home_team_team_runs_for_last5   0.007172743574417215
home_team_team_runs_for_last10   0.007172743574417215
home_team_team_runs_for_last20   0.007172743574417215
home_team_team_runs_against_season_to_date   0.007172743574417215
home_team_team_runs_against_last3   0.007172743574417215
home_team_team_runs_against_last5   0.007172743574417215
home_team_team_r

In [ ]:
num_cols = df_na.select_dtypes('float64','int64').columns
df_na[num_cols].corr()

,home_score,away_score,home_probable_pitcher_id,away_probable_pitcher_id,target_home_win,target_total_runs,target_home_margin,home_win,away_win,home_run_diff,away_run_diff,home_elo_pre,away_elo_pre,diff_elo_pre,elo_home_win_prob,home_team_team_runs_for_season_to_date,home_team_team_runs_for_last3,home_team_team_runs_for_last5,home_team_team_runs_for_last10,home_team_team_runs_for_last20,home_team_team_runs_against_season_to_date,home_team_team_runs_against_last3,home_team_team_runs_against_last5,home_team_team_runs_against_last10,home_team_team_runs_against_last20,home_team_team_win_season_to_date,home_team_team_win_last3,home_team_team_win_last5,home_team_team_win_last10,home_team_team_win_last20,home_team_team_run_diff_season_to_date,home_team_team_run_diff_last3,home_team_team_run_diff_last5,home_team_team_run_diff_last10,home_team_team_run_diff_last20,away_team_team_runs_for_season_to_date,away_team_team_runs_for_last3,away_team_team_runs_for_last5,away_team_team_runs_for_last10,away_team_team_runs_for_last20,away_team_team_runs_against_season_to_date,away_team_team_runs_against_last3,away_team_team_runs_against_last5,away_team_team_runs_against_last10,away_team_team_runs_against_last20,away_team_team_win_season_to_date,away_team_team_win_last3,away_team_team_win_last5,away_team_team_win_last10,away_team_team_win_last20,away_team_team_run_diff_season_to_date,away_team_team_run_diff_last3,away_team_team_run_diff_last5,away_team_team_run_diff_last10,away_team_team_run_diff_last20,diff_team_team_runs_for_season_to_date,diff_team_team_runs_for_last3,diff_team_team_runs_for_last5,diff_team_team_runs_for_last10,diff_team_team_runs_for_last20,diff_team_team_runs_against_season_to_date,diff_team_team_runs_against_last3,diff_team_team_runs_against_last5,diff_team_team_runs_against_last10,diff_team_team_runs_against_last20,diff_team_team_win_season_to_date,diff_team_team_win_last3,diff_team_team_win_last5,diff_team_team_win_last10,diff_team_team_win_last20,diff_team_team_run_diff_season_to_date,diff_team_team_run_diff_last3,diff_team_team_run_diff_last5,diff_team_team_run_diff_last10,diff_team_team_run_diff_last20,home_box_box_runs_for_season_to_date,home_box_box_runs_for_last3,home_box_box_runs_for_last5,home_box_box_runs_for_last10,home_box_box_runs_for_last20,home_box_box_runs_against_season_to_date,home_box_box_runs_against_last3,home_box_box_runs_against_last5,home_box_box_runs_against_last10,home_box_box_runs_against_last20,home_box_box_box_bat_flyOuts_season_to_date,home_box_box_box_bat_flyOuts_last3,home_box_box_box_bat_flyOuts_last5,home_box_box_box_bat_flyOuts_last10,home_box_box_box_bat_flyOuts_last20,home_box_box_box_bat_groundOuts_season_to_date,home_box_box_box_bat_groundOuts_last3,home_box_box_box_bat_groundOuts_last5,home_box_box_box_bat_groundOuts_last10,home_box_box_box_bat_groundOuts_last20,home_box_box_box_bat_airOuts_season_to_date,home_box_box_box_bat_airOuts_last3,home_box_box_box_bat_airOuts_last5,home_box_box_box_bat_airOuts_last10,home_box_box_box_bat_airOuts_last20,home_box_box_box_bat_runs_season_to_date,home_box_box_box_bat_runs_last3,home_box_box_box_bat_runs_last5,home_box_box_box_bat_runs_last10,home_box_box_box_bat_runs_last20,home_box_box_box_bat_doubles_season_to_date,home_box_box_box_bat_doubles_last3,home_box_box_box_bat_doubles_last5,home_box_box_box_bat_doubles_last10,home_box_box_box_bat_doubles_last20,home_box_box_box_bat_triples_season_to_date,home_box_box_box_bat_triples_last3,home_box_box_box_bat_triples_last5,home_box_box_box_bat_triples_last10,home_box_box_box_bat_triples_last20,home_box_box_box_bat_homeRuns_season_to_date,home_box_box_box_bat_homeRuns_last3,home_box_box_box_bat_homeRuns_last5,home_box_box_box_bat_homeRuns_last10,home_box_box_box_bat_homeRuns_last20,home_box_box_box_bat_strikeOuts_season_to_date,home_box_box_box_bat_strikeOuts_last3,home_box_box_box_bat_strikeOuts_last5,home_box_box_box_bat_strikeOuts_last10,home_box_box_box_bat_strikeOuts_last20,...,away_starter_starter